In [66]:
import tqdm
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.model_selection import KFold

In [67]:
df_origin = pd.read_csv("course_lead_scoring_2026.csv")

In [68]:
#Data preparation
def data_prepa(df):
    categorical_features = list((df.dtypes[df.dtypes == 'str']).index)
    numerical_features = list((df.dtypes[df.dtypes != 'str']).index)

    return categorical_features,numerical_features

In [69]:
categorical_features = ['lead_source']
numerical_features = ['number_of_courses_viewed', 'annual_income']

In [70]:
df = df_origin[['lead_source','number_of_courses_viewed', 'annual_income','converted']]
df.head()

,lead_source,number_of_courses_viewed,annual_income,converted
0,organic_search,3,88160.0,1
1,social_media,3,72688.0,1
2,referral,3,44697.0,1
3,paid_ads,3,NaN,0
4,referral,4,24062.0,1


In [71]:
"""
If there are missing values:
For categorical features, replace them with 'NA'
For numerical features, replace with with 0.0 
"""
df[categorical_features] = df[categorical_features].fillna("NA")

df[numerical_features] = df[numerical_features].fillna(0)

In [72]:
#Splitting the data into 3 parts, train, test, val
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [73]:
df_full_train.head()

,lead_source,number_of_courses_viewed,annual_income,converted
1233,paid_ads,0,38757.0,0
1056,paid_ads,2,56516.0,1
1686,paid_ads,1,54963.0,0
187,social_media,1,26237.0,0
3840,organic_search,3,21992.0,1


In [74]:
#building train function 
def train_model(df_train,C):
    y_train = df_train.converted.values
    X_train = df_train[categorical_features+numerical_features].reset_index(drop=True)\
        .to_dict(orient="records")

    dv = DictVectorizer(sparse=False)
    train_dicts = dv.fit_transform(X_train)

    model = LogisticRegression(solver='liblinear')
    model.fit(train_dicts,y_train)

    return model,dv

In [75]:
#doing predictions
def predict(df_val,model,dv):
    X = df_val[categorical_features+numerical_features].reset_index(drop=True)\
    .to_dict(orient="records")

    X_val = dv.transform(X)
    y_pred = model.predict_proba(X_val)[:,1]
    y_val = df_val.converted.values

    return y_pred,y_val

In [76]:
kfold = KFold(n_splits=5,shuffle=True,random_state=1)

scores = []
C = 1.0
for train_idx,val_idx in kfold.split(df_full_train):
    #separating train and val dataset by each fold
    fold_train = df_full_train.iloc[train_idx]
    fold_val = df_full_train.iloc[val_idx]

    model,dv = train_model(fold_train,C = C)
    fold_predictions,y_val = predict(fold_val,model,dv)

    auc = roc_auc_score(y_val,fold_predictions)
    scores.append(auc)

print('C=%s %.3f +- %.3f' % (C,np.mean(scores),np.std(scores)))
    

C=1.0 0.612 +- 0.019


In [77]:
scores

[0.5892665703739367,
 0.635455117249792,
 0.5952334980949439,
 0.6310128482804004,
 0.6076568313352625]

Save the model

In [78]:
import pickle

In [79]:
# output_file = "model_C = %s.bin"%C
output_file = f"model_C = {C}.bin"
output_file

'model_C = 1.0.bin'

In [80]:
f_out = open(output_file,"wb")
pickle.dump((model,dv),f_out)
f_out.close()#don't forget to close file

In [81]:
with open(output_file,"wb") as f_out:
    pickle.dump((model,dv),f_out)

Load the model

In [82]:
import pickle

In [83]:
output_file = 'model_C = 1.0.bin'

In [84]:
with open(output_file,"rb") as f_in:
   (model,dv)=pickle.load(f_in)

In [85]:
model,dv

(LogisticRegression(solver='liblinear'), DictVectorizer(sparse=False))

In [86]:
student = {
    "lead_source": "paid_ads",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0
}

In [87]:
X = dv.transform(student)
X

array([[7.9276e+04, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00,
        0.0000e+00, 0.0000e+00, 2.0000e+00]])

In [88]:
model.predict_proba(X)[0,1]

np.float64(0.653245700485014)

In [92]:
client = {
    "lead_source": "organic_search",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0
}


In [93]:
X_c = dv.transform(client)
X_c

array([[8.0304e+04, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 4.0000e+00]])

In [94]:
model.predict_proba(X_c)[0,1]

np.float64(0.6551075632763576)